# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayyankarar18/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Scoring.** Out of the four, scoring is the closest fit. What I'm making isn't a yes/no label (classification), and I'm not grouping pages into unlabeled buckets either (clustering). I'm giving each candidate signal a number that says how strongly it moves with performance. That's a scoring job — just done on signals instead of on content items. Once I know which signals actually matter, that's what a real content-scoring model (Lane 2 or 4) would be built on.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I'm scoring a signal, not a row, so there's no per row label to predict here. The score is an effect size, built from two columns that already exist in the data. An outcome like `scroll_rate` or `engagement_rate`, split by a candidate signal like `content_type` or `position_tier`. Both sides come straight from real, observed numbers. I'm not defining anything myself, so there's no risk of the label leakage problem you'd get with something like `is_declining_label`.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Effect size (Cohen's d).** A correlation just tells me two things move together. Cohen's d tells me how big the gap actually is, on a scale anyone can understand (small ~0.2, medium ~0.5, large ~0.8+). Below, splitting `scroll_rate` by `content_type` gives d = 1.71 between comparison articles and keyword articles. That's a big, real gap — not just noise from a small sample.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1) * a.std()**2 + (nb - 1) * b.std()**2) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled_std

comp = df[df["content_type"] == "comparison article"]["scroll_rate"]
kw = df[df["content_type"] == "keyword article"]["scroll_rate"]

print(f"comparison article mean scroll_rate: {comp.mean():.2f} (n={len(comp)})")
print(f"keyword article mean scroll_rate: {kw.mean():.2f} (n={len(kw)})")
print(f"Cohen's d: {cohens_d(comp, kw):.3f}")

comparison article mean scroll_rate: 61.82 (n=697)
keyword article mean scroll_rate: 15.48 (n=27207)
Cohen's d: 1.709


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content item (`content_id`), same as the full starter dataset. My slice just keeps the ID columns, the candidate signals, and the outcome metrics I'm checking them against. There's no per-row target to show, because I'm not scoring rows — I'm scoring signals. That score (Cohen's d) comes from grouping this same dataframe, which is what the code in section 3 already does.

In [7]:
lane_slice = df[[
    "content_id", "client_id", "content_type", "position_tier",
    "word_count", "scroll_rate", "engagement_rate", "ctr"
]]
print(lane_slice.shape)
lane_slice.head()

(30000, 8)


,content_id,client_id,content_type,position_tier,word_count,scroll_rate,engagement_rate,ctr
0,content_304f48230142,client_f369cb89fc,keyword article,striking,3221.0,4.55,5.88,0.76
1,content_a1fb4e703a9e,client_4e07408562,keyword article,page_3_5,2481.0,10.00,0.00,0.05
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,page_3_5,3515.0,28.57,0.00,0.09
3,content_331d6c4de07b,client_19581e27de,keyword article,page_1,NaN,3.45,1.28,0.49
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,page_3_5,2803.0,24.29,0.00,0.13


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A full trained model isn't needed for this step, a grouped comparison is what actually gives me the score. What a fixed rule can't handle is how many slices there are. 32 clients times 5 content types times several position tiers means the same signal can point different ways depending on which slice you're in. A rule like long content always scrolls better, falls apart the moment you check it per content type. That's too many combinations to hardcode by hand, so it needs a systematic check across all of them instead of one if-statement. This step is also what tells me later whether it's even worth building a real scoring model in Lane 2.

In [8]:
n_clients = df["client_id"].nunique()
n_types = df["content_type"].nunique()
n_tiers = df["position_tier"].nunique()
print(f"{n_clients} clients x {n_types} content_types x {n_tiers} position_tiers "
      f"= up to {n_clients * n_types * n_tiers} strata")

32 clients x 3 content_types x 5 position_tiers = up to 480 strata


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.